# Forward PINN for Darcy Flow Below a Reservoir

Goal: build a **forward Physics-Informed Neural Network** for the hydraulic head field `h(x, y)`.

Important distinction:

- The Excel files are useful for **visualizing the problem** and later checking your result.
- The forward PINN loss should use **only the PDE and boundary conditions**, not the measured/generated head values.

The notebook contains a complete PyTorch implementation of the forward PINN.


## 0. Imports

Use `numpy/pandas/matplotlib` for data inspection and plotting. Use `torch` later for automatic differentiation.


In [ ]:
# If needed, install once:
%pip install -q numpy pandas matplotlib openpyxl torch

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import torch
from torch import nn

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True


## 1. Choose a Case and Inspect the Provided Data

The filename pattern is:

```text
h1_h2_hd.xlsx
```

where:

- `h1` is the reservoir head,
- `h2` is the downstream catchment head,
- `hd` is the dam embedment depth.

For a forward PINN, we do **not** train on `FINIT`; here we only inspect it so we understand the geometry and have something to compare with after training.


In [ ]:
# This path assumes the notebook is opened from the Code/ folder.
# If your kernel starts from the project root, use: "Data/heads/20_5_25.xlsx"
data_path = "../Data/heads/20_5_25.xlsx"

h1, h2, hd = map(float, data_path.split("/")[-1].replace(".xlsx", "").split("_"))
df = pd.read_excel(data_path).dropna()

print(f"case: h1={h1:g} m, h2={h2:g} m, hd={hd:g} m")
print(df.head())
print(df[["X", "Y", "FINIT"]].describe())


## 2. Plot the Reference Head Data

Again: this is **not training data** for the forward PINN. It is a useful reference plot so you can later judge whether the physics-only solution has the correct shape.


In [ ]:
# Interpolate the irregular reference points with a triangular contour mesh.
x_ref = df["X"].to_numpy(float)
y_ref = df["Y"].to_numpy(float)
h_ref = df["FINIT"].to_numpy(float)
triangulation = mtri.Triangulation(x_ref, y_ref)

# Do not interpolate through the solid dam cutout.
triangles = triangulation.triangles
xc = x_ref[triangles].mean(axis=1)
yc = y_ref[triangles].mean(axis=1)
dam_triangle = ((xc >= 100.0) & (xc <= 120.0) & (yc >= 40.0 - hd))
triangulation.set_mask(dam_triangle)

fig, ax = plt.subplots(figsize=(9, 4))
levels = np.linspace(h_ref.min(), h_ref.max(), 100)
cf = ax.tricontourf(triangulation, h_ref, levels=levels, cmap="viridis", extend="both")
ax.tricontour(triangulation, h_ref, levels=12, colors="white", linewidths=0.45, alpha=0.45)
ax.fill([100, 120, 120, 100], [40-hd, 40-hd, 40, 40], color="0.65", label="impermeable dam")
fig.colorbar(cf, ax=ax, label="head h [m]")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("Provided head field: continuous reference interpolation")
ax.set_aspect("equal", adjustable="box")
plt.show()


## 3. Nondimensionalize the Governing Equation

The dimensional PDE is Laplace's equation:

$$
\Delta h = \frac{\partial^2 h}{\partial x^2} + \frac{\partial^2 h}{\partial y^2} = 0.
$$

Choose one length scale, for example the total horizontal length:

$$
L = 150\;\text{m}.
$$

Define dimensionless coordinates and head:

$$
\hat{x}=\frac{x}{L}, \qquad \hat{y}=\frac{y}{L}, \qquad
H = \frac{h-h_2}{h_1-h_2}.
$$

Then:

$$
h = h_2 + (h_1-h_2)H.
$$

Using the same length scale in both directions gives

$$
\frac{\partial h}{\partial x_i}=\frac{h_1-h_2}{L}\frac{\partial H}{\partial \hat{x}_i},\qquad
\frac{\partial^2 h}{\partial x_i^2}=\frac{h_1-h_2}{L^2}\frac{\partial^2 H}{\partial \hat{x}_i^2}.
$$

The nonzero constant factor $(h_1-h_2)/L^2$ cancels, so the PDE becomes:

$$
\frac{\partial^2 H}{\partial \hat{x}^2} +
\frac{\partial^2 H}{\partial \hat{y}^2}=0.
$$

The domain becomes $0\leq\hat{x}\leq1$ and $0\leq\hat{y}\leq40/150$. The excluded solid dam is $100/150\leq\hat{x}\leq120/150$ and $15/150\leq\hat{y}\leq40/150$.

The Dirichlet conditions become:

$$
H=1 \quad \text{on }0\leq\hat{x}\leq100/150,\;\hat{y}=40/150,
$$

$$
H=0 \quad \text{on }120/150\leq\hat{x}\leq1,\;\hat{y}=40/150.
$$

For an impermeable boundary, Darcy's law requires $v_n=-k_f\nabla h\cdot n=0$. Because

$$
\nabla h\cdot n=\frac{h_1-h_2}{L}\nabla_{\hat{x},\hat{y}}H\cdot n,
$$

the nondimensional no-flow condition is

$$
\nabla_{\hat{x},\hat{y}}H\cdot n=0.
$$

Thus `vx0` enforces $H_{,\hat{x}}=0$ on vertical impermeable segments and `vy0` enforces $H_{,\hat{y}}=0$ on the bedrock and dam underside. The top of the solid dam is outside the water domain and receives no boundary condition.


In [ ]:
y_top = 40.0
y_bot = 0.0
x_left = 0.0
x_right = 150.0
x_dleft = 100.0
x_dright = 120.0
y_dbot = y_top - hd
num_bc = 100

# Existing geometry names retained for the nondimensionalization and plots.
L = x_right - x_left
D = y_top - y_bot
x_res = x_dleft
x_dam = x_dright
kf = 1.0       # hydraulic conductivity [m/day]
head_scale = h1 - h2
if head_scale == 0:
    raise ValueError("h1 and h2 must differ for head nondimensionalization")

def to_hat_xy(xy):
    """Convert dimensional coordinates [m] to nondimensional coordinates."""
    return np.asarray(xy) / L

def to_hat_head(h):
    """Convert dimensional hydraulic head h [m] to H."""
    return (h - h2) / head_scale

def from_hat_head(H):
    """Convert nondimensional head H back to dimensional head h [m]."""
    return h2 + head_scale * H

print("Domain in nondimensional coordinates:")
print(f"x_hat in [0, {L/L:.3f}]")
print(f"y_hat in [0, {D/L:.3f}]")
print(f"dam cutout: x_hat in [{x_dleft/L:.3f}, {x_dright/L:.3f}], "
      f"y_hat in [{y_dbot/L:.3f}, {y_top/L:.3f}]")
print(f"Dirichlet targets: H(h1)={to_hat_head(h1):.1f}, H(h2)={to_hat_head(h2):.1f}")
print(f"gradient conversion: grad(h) = {head_scale/L:.3f} grad_hat(H)")


## 4. Draw the Domain and Boundary Conditions

This plot is the checklist for your loss terms.

Suggested boundary conditions:

- Top-left reservoir surface: `H = 1`
- Top-right catchment surface: `H = 0`
- Bottom bedrock: `dH/dy = 0`
- Left and right vertical boundaries: `dH/dx = 0`
- Dam underside from `x=100` to `x=120`: `dH/dy = 0`
- Both vertical dam faces: `dH/dx = 0`

The solid dam rectangle is excluded from the PDE domain; in particular, the line from `x=100` to `x=120` at the top is not a water-accessible boundary.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Aquifer perimeter.  The dam body is excluded from the flow domain.
y_dam_base = D - hd
domain_x = [0, L, L, x_dam, x_dam, x_res, x_res, 0, 0]
domain_y = [0, 0, D, D, y_dam_base, y_dam_base, D, D, 0]
ax.plot(domain_x, domain_y, color="black", lw=1.5)
ax.fill([x_res, x_dam, x_dam, x_res], [y_dam_base, y_dam_base, D, D],
        color="0.75", label="Impermeable dam (outside domain)")

# Dirichlet boundaries
ax.plot([0, x_res], [D, D], color="tab:blue", lw=4, label="Reservoir: H=1")
ax.plot([x_dam, L], [D, D], color="tab:orange", lw=4, label="Catchment: H=0")

# Impermeable aquifer--dam interface: both faces and the underside.
ax.plot([x_res, x_res], [y_dam_base, D], color="crimson", lw=4, label="No-flow dam interface")
ax.plot([x_res, x_dam], [y_dam_base, y_dam_base], color="crimson", lw=4)
ax.plot([x_dam, x_dam], [y_dam_base, D], color="crimson", lw=4)

ax.text(35, D + 1.2, "H=1", color="tab:blue", ha="center")
ax.text(135, D + 1.2, "H=0", color="tab:orange", ha="center")
ax.text((x_res + x_dam) / 2, y_dam_base - 1.2, "dH/dn=0", color="crimson", ha="center", va="top")
ax.text(70, 2, "dH/dy=0", color="black", ha="center")

ax.set_xlim(-5, L + 5)
ax.set_ylim(-3, D + 6)
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="lower left", ncols=2)
ax.set_title("Domain and boundary-condition map")
plt.show()


## 5. Sample Collocation and Boundary Points

The PINN needs points where each loss is evaluated:

- Interior collocation points for the PDE residual
- Boundary points for Dirichlet constraints
- Boundary points for Neumann/no-flow constraints

The code below rejects points inside the solid dam. You can improve the sampler later, for example by adding extra points near the two lower dam corners where the solution changes quickly.


In [ ]:
rng = np.random.default_rng(7)

def _line_count(length, n_density):
    """Convert points-per-metre density to a valid linspace count."""
    if n_density <= 0:
        raise ValueError("n_density must be positive")
    return max(2, int(round(n_density * length)))

def sample_boundaries(n_density):
    """Sample Dirichlet and no-flow portions of the notched perimeter."""
    x1 = np.linspace(x_left, x_dleft, _line_count(x_dleft - x_left, n_density))
    h1_boundary = np.column_stack([x1, np.full_like(x1, y_top)])

    x2 = np.linspace(x_dright, x_right, _line_count(x_right - x_dright, n_density))
    h2_boundary = np.column_stack([x2, np.full_like(x2, y_top)])

    y3 = np.linspace(y_dbot, y_top, _line_count(y_top - y_dbot, n_density))
    dam_left = np.column_stack([np.full_like(y3, x_dleft), y3])
    dam_right = np.column_stack([np.full_like(y3, x_dright), y3])

    y_side = np.linspace(y_bot, y_top, _line_count(y_top - y_bot, n_density))
    right = np.column_stack([np.full_like(y_side, x_right), y_side])
    left = np.column_stack([np.full_like(y_side, x_left), y_side])
    vx0 = np.vstack([dam_left, dam_right, right, left])

    x5 = np.linspace(x_dleft, x_dright, _line_count(x_dright - x_dleft, n_density))
    dam_base = np.column_stack([x5, np.full_like(x5, y_dbot)])
    x6 = np.linspace(x_left, x_right, _line_count(x_right - x_left, n_density))
    bottom = np.column_stack([x6, np.full_like(x6, y_bot)])
    vy0 = np.vstack([dam_base, bottom])

    return {"h1": h1_boundary, "h2": h2_boundary, "vx0": vx0, "vy0": vy0}

def sample_interior_points(num_points=1000):
    """Sample the aquifer and reject candidates inside the solid dam."""
    points = []
    while len(points) < num_points:
        x_candidate = rng.uniform(x_left, x_right)
        y_candidate = rng.uniform(y_bot, y_top)
        inside_dam = (x_dleft <= x_candidate <= x_dright and
                      y_dbot <= y_candidate <= y_top)
        if not inside_dam:
            points.append((x_candidate, y_candidate))
    points = np.asarray(points)
    return points[:, 0], points[:, 1]

def sample_interior(num_points=1000):
    x, y = sample_interior_points(num_points)
    return np.column_stack([x, y])

xy_int = sample_interior(800)
n_density = 10
bc = sample_boundaries(n_density)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(xy_int[:, 0], xy_int[:, 1], s=6, alpha=0.25, label="PDE points")
ax.scatter(bc["h1"][:, 0], bc["h1"][:, 1], s=8, label="h1: H=1")
ax.scatter(bc["h2"][:, 0], bc["h2"][:, 1], s=8, label="h2: H=0")
ax.scatter(bc["vx0"][:, 0], bc["vx0"][:, 1], s=6, label="vx0: dH/dx=0")
ax.scatter(bc["vy0"][:, 0], bc["vy0"][:, 1], s=6, label="vy0: dH/dy=0")
ax.fill([x_res, x_dam, x_dam, x_res], [D-hd, D-hd, D, D], color="0.75")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
ax.set_title("Sampled PINN training points")
plt.show()


## 6. Plan the PINN Loss

Your model receives nondimensional coordinates and predicts nondimensional head:

$$
(\hat{x}, \hat{y}) \mapsto H_\theta(\hat{x}, \hat{y}).
$$

The total loss can be:

$$
\mathcal{L} = \mathcal{L}_{PDE} + \lambda_D \mathcal{L}_{D} + \lambda_N \mathcal{L}_{N}.
$$

PDE loss:

$$
\mathcal{L}_{PDE} = \text{MSE}\left(
\frac{\partial^2 H_\theta}{\partial \hat{x}^2} +
\frac{\partial^2 H_\theta}{\partial \hat{y}^2}, 0
\right).
$$

Dirichlet loss:

$$
\mathcal{L}_{D}=\text{MSE}(H_\theta, 1)_{reservoir}
+\text{MSE}(H_\theta, 0)_{catchment}.
$$

Neumann/no-flow loss:

$$
\mathcal{L}_{N}=\text{MSE}(\nabla H_\theta \cdot n, 0)_{impermeable}.
$$


## 7. PyTorch Setup: You Fill in the Network

Start simple: an MLP with `Tanh` activations usually works well for PINNs.

Suggested architecture:

```text
input:  (x_hat, y_hat)
width:  32 or 64
layers: 4 to 6 hidden layers
output: H_hat
```

Implementation hints:

- Convert sampled numpy arrays to `torch.float32` tensors.
- Use `requires_grad_(True)` on coordinates when computing derivatives.
- Use `torch.autograd.grad(..., create_graph=True)` for first and second derivatives.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

torch.manual_seed(7)

class HeadPINN(nn.Module):
    def __init__(self, hidden_width=64, hidden_layers=4):
        super().__init__()
        layers = [nn.Linear(2, hidden_width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(hidden_width, hidden_width), nn.Tanh()])
        layers.append(nn.Linear(hidden_width, 1))
        self.net = nn.Sequential(*layers)
        self.net.apply(self._init_weights)

    @staticmethod
    def _init_weights(layer):
        if isinstance(layer, nn.Linear):
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, xy_hat):
        return self.net(xy_hat)


## 8. Autograd Helpers

This is the key PINN trick. You need derivatives of the network output with respect to its input coordinates.

Fill in the missing pieces after you define the model.


In [ ]:
def torch_xy(xy_np):
    xy_hat_np = to_hat_xy(xy_np)
    return torch.tensor(xy_hat_np, dtype=torch.float32, device=device)

def grad(outputs, inputs):
    return torch.autograd.grad(
        outputs=outputs,
        inputs=inputs,
        grad_outputs=torch.ones_like(outputs),
        create_graph=True,
        retain_graph=True
    )[0]

def laplace_residual(model, xy_hat):
    """Dimensionless residual H_xhatxhat + H_yhatyhat."""
    xy_hat = xy_hat.clone().detach().requires_grad_(True)

    H = model(xy_hat)

    grad_H = grad(H, xy_hat)

    H_x = grad_H[:, 0:1]
    H_y = grad_H[:, 1:2]

    grad_H_x = grad(H_x, xy_hat)
    grad_H_y = grad(H_y, xy_hat)

    H_xx = grad_H_x[:, 0:1]
    H_yy = grad_H_y[:, 1:2]

    return H_xx + H_yy

def normal_derivative(model, xy_hat, component):
    """Dimensionless coordinate derivative used for homogeneous no-flow BCs."""
    xy_hat = xy_hat.clone().detach().requires_grad_(True)

    H = model(xy_hat)

    grad_H = grad(H, xy_hat)

    return grad_H[:, component:component+1]


## 9. Build One Training Step

Do this before writing a long training loop. A single step should:

1. Sample points.
2. Convert them to torch tensors.
3. Compute PDE, Dirichlet, and Neumann losses.
4. Combine them.
5. Backpropagate and update the model.

Weights such as `lambda_D = 20` and `lambda_N = 5` are starting points, not sacred constants.


In [ ]:
model = HeadPINN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
lambda_D = 20.0
lambda_N = 5.0

def training_step(n_int=2048, n_density=2):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    xy_int = torch_xy(sample_interior(n_int))
    bc_np = sample_boundaries(n_density)

    pde = laplace_residual(model, xy_int)
    loss_pde = torch.mean(pde**2)

    H_res = model(torch_xy(bc_np["h1"]))
    H_cat = model(torch_xy(bc_np["h2"]))
    loss_D = torch.mean((H_res - 1.0)**2) + torch.mean(H_cat**2)

    dH_dx_zero = normal_derivative(model, torch_xy(bc_np["vx0"]), component=0)
    dH_dy_zero = normal_derivative(model, torch_xy(bc_np["vy0"]), component=1)
    loss_N = torch.mean(dH_dx_zero**2) + torch.mean(dH_dy_zero**2)

    loss = loss_pde + lambda_D * loss_D + lambda_N * loss_N
    loss.backward()
    optimizer.step()

    return {
        "total": loss.item(),
        "pde": loss_pde.item(),
        "dirichlet": loss_D.item(),
        "neumann": loss_N.item(),
    }


## 10. Training Loop and Loss Plot

Once one training step works, run the loop and plot the losses. If one loss term dominates, tune `lambda_D`, `lambda_N`, the sample counts, or the learning rate.


In [ ]:
n_epochs = 3000
history = []
for epoch in range(1, n_epochs + 1):
    logs = training_step()
    history.append(logs)

    if epoch == 1 or epoch % 300 == 0:
        print(epoch, logs)

hist = pd.DataFrame(history)
ax = hist.plot(logy=True, figsize=(8, 4))
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("PINN training losses")
plt.show()


## 11. Plot the Learned Head Field

After training, evaluate the model on a grid, convert `H` back to dimensional `h`, and draw contours.


In [ ]:
nx, ny = 160, 70
xs = np.linspace(0, L, nx)
ys = np.linspace(0, D, ny)
X, Y = np.meshgrid(xs, ys)
grid_xy = np.column_stack([X.ravel(), Y.ravel()])
dam_mask = ((X >= x_res) & (X <= x_dam) & (Y >= D - hd))

model.eval()
with torch.no_grad():
    H_pred = model(torch_xy(grid_xy)).cpu().numpy().reshape(ny, nx)
h_pred = from_hat_head(H_pred)
h_pred[dam_mask] = np.nan

fig, ax = plt.subplots(figsize=(10, 4))
cf = ax.contourf(X, Y, h_pred, levels=40, cmap="viridis")
ax.contour(X, Y, h_pred, levels=12, colors="white", alpha=0.5, linewidths=0.7)
ax.fill([x_res, x_dam, x_dam, x_res], [D-hd, D-hd, D, D], color="0.6")
fig.colorbar(cf, ax=ax, label="predicted head h [m]")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_aspect("equal", adjustable="box")
ax.set_title("PINN-predicted head field")
plt.show()


## 12. Optional Validation Against the Spreadsheet

This is not part of the training objective. Use it only after the physics-only model is trained.


In [ ]:
ref_xy = df[["X", "Y"]].to_numpy(float)
h_true = df["FINIT"].to_numpy(float)

model.eval()
with torch.no_grad():
    H_ref = model(torch_xy(ref_xy)).cpu().numpy().ravel()
h_ref_pred = from_hat_head(H_ref)

rmse = np.sqrt(np.mean((h_ref_pred - h_true)**2))
mae = np.mean(np.abs(h_ref_pred - h_true))
print(f"RMSE: {rmse:.3f} m")
print(f"MAE:  {mae:.3f} m")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(h_true, h_ref_pred, s=18, alpha=0.7)
lims = [min(h_true.min(), h_ref_pred.min()), max(h_true.max(), h_ref_pred.max())]
ax.plot(lims, lims, color="black", lw=1)
ax.set_xlabel("reference h [m]")
ax.set_ylabel("PINN h [m]")
ax.set_title("Reference comparison, not training")
ax.set_aspect("equal", adjustable="box")
plt.show()


## 13. Optional Flow Rate Estimate

Using Darcy's law with the usual sign convention:

$$
v_d = -k_f \nabla h.
$$

Since

$$
h = h_2 + (h_1-h_2)H,
$$

then

$$
\nabla h = \frac{h_1-h_2}{L}\nabla_{\hat{x},\hat{y}} H.
$$

For the downstream top boundary, estimate discharge per metre width by integrating the vertical velocity:

$$
Q = \int_{x=120}^{150} v_y\,dx.
$$

The following cell evaluates this quantity after training.


In [ ]:
x_line = np.linspace(x_dam, L, 400)
top_line = np.column_stack([x_line, np.full_like(x_line, D)])
xy_hat = torch_xy(top_line).requires_grad_(True)
H_top = model(xy_hat)
grad_H = grad(H_top, xy_hat)
dH_dy_hat = grad_H[:, 1].detach().cpu().numpy()

vy = -kf * head_scale / L * dH_dy_hat
Q = np.trapezoid(vy, x_line)
print(f"Q = {Q:.4f} m^2/day per metre width")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(x_line, vy)
ax.set_xlabel("x [m]")
ax.set_ylabel("vertical Darcy velocity [m/day]")
ax.set_title("Downstream outflow velocity profile")
plt.show()
